Parametrix for a Pseudodifferential Operator

Let P be a ψDO with symbol p(x, ξ). An asymptotic right‑parametrix Q satisfies
P ∘ Q = I + R, where R is a smoothing operator (symbol of order ‑∞).
In the semi‑classical setting with large parameter λ, one constructs the symbol of Q as an asymptotic series in λ⁻¹.

The psiop module (imported in fio_bridge.py) provides the method
PseudoDifferentialOperator.inverse_asymptotic(order, mode) to obtain the symbol of the parametrix.

## Steps to build and evaluate the parametrix

In [ ]:
import sympy as sp
import numpy as np
from psiop import PseudoDifferentialOperator
from fio_bridge import PsiOpFIOBridge

# 1. Define operator with explicit potential
x, xi = sp.symbols('x xi', real=True)
V_expr = x**2                     # concrete function, not a symbolic Function
p_symbol = xi**2 + V_expr

P = PseudoDifferentialOperator(p_symbol, vars_x=[x], mode='symbol')

# 2. Compute parametrix symbol and evaluate derivatives
Q_symbol = P.right_inverse_asymptotic(order=2).doit()   # <-- critical

# 3. Wrap into PsiOpFIOBridge
lam = 50.0
bridge = PsiOpFIOBridge(
    PseudoDifferentialOperator(Q_symbol, vars_x=[x], mode='symbol'),
    lam=lam
)

# 4. Evaluate on a grid
x_vals = np.linspace(-10, 10, 1000)
u0_amp = sp.exp(-x**2 / 2)   # peak = 1
u0_phase = sp.Integer(0)     # no oscillation, or a linear phase like x

result = bridge.evaluate_grid(x_vals, u0_phase, u0_amp)

For a time‑dependent parametrix (the propagator), the package already provides PropagatorBridge, which constructs exp(itP) via the exponential symbol:

In [ ]:
from fio_bridge import PropagatorBridge

prop_bridge = PropagatorBridge(P, lam=lam, exp_order=2)
u_t = prop_bridge.propagate(t=0.1, x_grid=x_vals, u0_phase_sym=u0_phase, u0_amp_sym=u0_amp)


## Checking the Parametrix Quality

The CrossValidator class can compare the exact (spectral) action of P with the approximate action of its parametrix Q.
Set up a WKBState and run the validator:

In [ ]:
from fio_bridge import WKBState, CrossValidator

wkb = WKBState(u0_amp, u0_phase, var_x=x, lam=lam)
validator = CrossValidator(P, wkb, x_grid=x_vals, lam=lam,
                           bridge_kwargs={'n_guesses': 40})
report = validator.run()   # runs both P (via solver) and Q (via bridge)

# Plot the comparison
CrossValidator.plot_report(report)

In [ ]:
import matplotlib.pyplot as plt
x_vals = np.linspace(-5, 5, 200)
u0_num = np.exp(-x_vals**2) * np.exp(1j * 50 * (x_vals**2/2))
plt.plot(x_vals, np.abs(u0_num))
plt.title("|u0(x)|")
plt.show()

In [ ]:
from fio_bridge import CrossValidator, WKBState

wkb = WKBState(u0_amp, u0_phase, var_x=x, lam=50)
validator = CrossValidator(P, wkb, x_grid=x_vals, lam=50)

u_bridge = validator.run_bridge_only()
u_solver = validator.run_solver_only()

print("Bridge: max |u| =", np.max(np.abs(u_bridge)))
print("Solver: max |u| =", np.max(np.abs(u_solver)))

In [ ]:
import sympy as sp
import numpy as np
from psiop import PseudoDifferentialOperator
from fio_bridge import PsiOpFIOBridge, CrossValidator, WKBState

x, xi = sp.symbols('x xi', real=True)

# Identity operator
P = PseudoDifferentialOperator(sp.Integer(1), vars_x=[x], mode='symbol')

# WKB state: Gaussian amplitude, zero phase (so u0 is real, smooth)
u0_amp = sp.exp(-x**2/2)
u0_phase = sp.Integer(0)

lam = 20.0
x_grid = np.linspace(-5, 5, 400)   # fine grid

# Bridge evaluation
bridge = PsiOpFIOBridge(P, lam=lam, y_range=(-10,10), xi_range=(-15,15))
u_bridge = bridge.evaluate_grid(x_grid, u0_phase, u0_amp)

# Exact result: Pu = u (since P=I)
u_exact = np.exp(-x_grid**2/2)

plt.plot(x_grid, u_bridge.real, label='bridge')
plt.plot(x_grid, u_exact, '--', label='exact')
plt.legend()
plt.show()

In [ ]:
bridge = PsiOpFIOBridge(P, lam=lam, verbose=True)   # turn on verbose
result = bridge.evaluate_at(x_val=0.0, u_phase_sym=u0_phase, u_amp_sym=u0_amp)
print("Number of critical points:", result.n_critical_points)
print("Contributions:", result.contributions)